In [3]:
import tensorflow as tf
print("GPUs:", tf.config.list_physical_devices("GPU"))

a = tf.constant([1.0, 2.0, 3.0])
print(a * 2)

GPUs: [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]
tf.Tensor([2. 4. 6.], shape=(3,), dtype=float32)


## Setup

In [4]:
import numpy as np
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

SEED = 42
np.random.seed(SEED)
tf.random.set_seed(SEED)

print("TF version:", tf.__version__)

TF version: 2.19.0


In [5]:
import tensorflow as tf
print("GPUs:", tf.config.list_physical_devices("GPU"))

# Optional speedup on GPU:
from tensorflow.keras import mixed_precision
mixed_precision.set_global_policy("mixed_float16")
print("Policy:", mixed_precision.global_policy())

GPUs: [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]
Policy: <DTypePolicy "mixed_float16">


## Load Data

In [8]:
# --- CHANGE THESE PATHS IF NEEDED ---
X_PATH = "/content/X.npy"
Y_PATH = "/content/y.npy"
Z_PATH = "/content/z.npy"   # optional

X = np.load(X_PATH)
y = np.load(Y_PATH)

print("X:", X.shape, X.dtype)
print("y:", y.shape, y.dtype, "min/max:", y.min(), y.max())

# Optional outcome labels, not required for move-classification
try:
    z = np.load(Z_PATH)
    print("z:", z.shape, z.dtype, "unique:", np.unique(z)[:10])
except Exception:
    z = None
    print("z not loaded (ok).")

X: (92559, 6, 7, 2) int8
y: (92559,) int64 min/max: 0 6
z: (92559,) int64 unique: [-1  0  1]


## Train/Val/Test split, tf.data, and symmetry flip augmentation

In [9]:
import tensorflow as tf

AUTOTUNE = tf.data.AUTOTUNE

def flip_augment_cpu(x, y):
    # Keep everything CPU-friendly and simple
    do_flip = tf.random.uniform(()) < 0.5
    if_flip_x = tf.reverse(x, axis=[1])   # flip width
    if_flip_y = 6 - y
    x = tf.where(do_flip, if_flip_x, x)
    y = tf.where(do_flip, if_flip_y, y)
    return x, y

def make_ds(X, y, batch=256, training=True):
    with tf.device("/CPU:0"):
        ds = tf.data.Dataset.from_tensor_slices((X, y))
        if training:
            ds = ds.shuffle(min(len(X), 50_000), seed=42, reshuffle_each_iteration=True)
            ds = ds.map(flip_augment_cpu, num_parallel_calls=AUTOTUNE)
        ds = ds.batch(batch)
        ds = ds.prefetch(AUTOTUNE)
    return ds

# split indices
n = len(X)
idx = np.arange(n)
rng = np.random.default_rng(SEED)
rng.shuffle(idx)

train_end = int(0.85 * n)
val_end   = int(0.93 * n)

tr = idx[:train_end]
va = idx[train_end:val_end]
te = idx[val_end:]

BATCH = 256
train_ds = make_ds(X[tr], y[tr], batch=BATCH, training=True)
val_ds   = make_ds(X[va], y[va], batch=BATCH, training=False)
test_ds  = make_ds(X[te], y[te], batch=BATCH, training=False)

print("Splits:", len(tr), len(va), len(te))

Splits: 78675 7404 6480


## CNN Model

In [10]:
def build_cnn(input_shape=(6,7,2)):
    inp = keras.Input(shape=input_shape)

    x = layers.Conv2D(64, 3, padding="same", use_bias=False)(inp)
    x = layers.BatchNormalization()(x)
    x = layers.ReLU()(x)

    for filters, sd in [(64, 0.10), (96, 0.10), (128, 0.15)]:
        x = layers.Conv2D(filters, 3, padding="same", use_bias=False)(x)
        x = layers.BatchNormalization()(x)
        x = layers.ReLU()(x)
        x = layers.SpatialDropout2D(sd)(x)

        x = layers.Conv2D(filters, 3, padding="same", use_bias=False)(x)
        x = layers.BatchNormalization()(x)
        x = layers.ReLU()(x)

    x = layers.GlobalAveragePooling2D()(x)
    x = layers.Dense(128, activation="relu")(x)
    x = layers.Dropout(0.35)(x)

    out = layers.Dense(7, activation="softmax", dtype="float32")(x)
    return keras.Model(inp, out, name="cnn_connect4")

cnn = build_cnn()
cnn.compile(
    optimizer=keras.optimizers.Adam(learning_rate=3e-4),
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)
cnn.summary()

Model: "cnn_connect4"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer (InputLayer)        │ (None, 6, 7, 2)        │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d (Conv2D)                 │ (None, 6, 7, 64)       │         1,152 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization             │ (None, 6, 7, 64)       │           256 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ re_lu (ReLU)                    │ (None, 6, 7, 64)       │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_1 (Conv2D)               │ (None, 6, 7, 64)       │        36,864 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_1           │ (None, 6, 7, 64)       │           256 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ re_lu_1 (ReLU)                  │ (None, 6, 7, 64)       │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ spatial_dropout2d               │ (None, 6, 7, 64)       │             0 │
│ (SpatialDropout2D)              │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_2 (Conv2D)               │ (None, 6, 7, 64)       │        36,864 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_2           │ (None, 6, 7, 64)       │           256 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ re_lu_2 (ReLU)                  │ (None, 6, 7, 64)       │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_3 (Conv2D)               │ (None, 6, 7, 96)       │        55,296 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_3           │ (None, 6, 7, 96)       │           384 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ re_lu_3 (ReLU)                  │ (None, 6, 7, 96)       │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ spatial_dropout2d_1             │ (None, 6, 7, 96)       │             0 │
│ (SpatialDropout2D)              │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_4 (Conv2D)               │ (None, 6, 7, 96)       │        82,944 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_4           │ (None, 6, 7, 96)       │           384 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ re_lu_4 (ReLU)                  │ (None, 6, 7, 96)       │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_5 (Conv2D)               │ (None, 6, 7, 128)      │       110,592 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_5           │ (None, 6, 7, 128)      │           512 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ re_lu_5 (ReLU)                  │ (None, 6, 7, 128)      │             

 Total params: 491,143 (1.87 MB)

 Trainable params: 489,863 (1.87 MB)

 Non-trainable params: 1,280 (5.00 KB)

Train & Evaluate CNN Model

In [11]:
callbacks = [
    keras.callbacks.EarlyStopping(monitor="val_loss", patience=8, restore_best_weights=True),
    keras.callbacks.ReduceLROnPlateau(monitor="val_loss", factor=0.5, patience=3, min_lr=1e-6),
    keras.callbacks.ModelCheckpoint("best_cnn.keras", monitor="val_accuracy", save_best_only=True),
]

history_cnn = cnn.fit(
    train_ds,
    validation_data=val_ds,
    epochs=50,
    callbacks=callbacks
)

cnn_test = cnn.evaluate(test_ds, verbose=0)
print("CNN test loss/acc:", cnn_test)

cnn.save("cnn_final.keras")

Epoch 1/50
308/308 ━━━━━━━━━━━━━━━━━━━━ 67s 103ms/step - accuracy: 0.2899 - loss: 1.7543 - val_accuracy: 0.2473 - val_loss: 3.2117 - learning_rate: 3.0000e-04
Epoch 2/50
308/308 ━━━━━━━━━━━━━━━━━━━━ 2s 5ms/step - accuracy: 0.4547 - loss: 1.4538 - val_accuracy: 0.4922 - val_loss: 1.3903 - learning_rate: 3.0000e-04
Epoch 3/50
308/308 ━━━━━━━━━━━━━━━━━━━━ 2s 5ms/step - accuracy: 0.5096 - loss: 1.2914 - val_accuracy: 0.5451 - val_loss: 1.1809 - learning_rate: 3.0000e-04
Epoch 4/50
308/308 ━━━━━━━━━━━━━━━━━━━━ 2s 5ms/step - accuracy: 0.5387 - loss: 1.2117 - val_accuracy: 0.5720 - val_loss: 1.1262 - learning_rate: 3.0000e-04
Epoch 5/50
308/308 ━━━━━━━━━━━━━━━━━━━━ 2s 5ms/step - accuracy: 0.5595 - loss: 1.1599 - val_accuracy: 0.5898 - val_loss: 1.0726 - learning_rate: 3.0000e-04
Epoch 6/50
308/308 ━━━━━━━━━━━━━━━━━━━━ 2s 5ms/step - accuracy: 0.5743 - loss: 1.1102 - val_accuracy: 0.5922 - val_loss: 1.0579 - learning_rate: 3.0000e-04
Epoch 7/50
308/308 ━━━━━━━━━━━━━━━━━━━━ 2s 5ms/step - accurac

## Transformer Model (42 cell tokens)

In [13]:
class TransformerBlock(layers.Layer):
    def __init__(self, embed_dim, num_heads, mlp_dim, dropout=0.1):
        super().__init__()
        self.ln1 = layers.LayerNormalization()
        self.attn = layers.MultiHeadAttention(num_heads=num_heads, key_dim=embed_dim)
        self.drop1 = layers.Dropout(dropout)

        self.ln2 = layers.LayerNormalization()
        self.mlp = keras.Sequential([
            layers.Dense(mlp_dim, activation="gelu"),
            layers.Dropout(dropout),
            layers.Dense(embed_dim),
            layers.Dropout(dropout),
        ])

    def call(self, x, training=False):
        h = self.ln1(x)
        h = self.attn(h, h, training=training)
        x = x + self.drop1(h, training=training)
        x = x + self.mlp(self.ln2(x), training=training)
        return x

def build_transformer(input_shape=(6,7,2), embed_dim=64, num_heads=4, mlp_dim=128, depth=4):
    inp = keras.Input(shape=input_shape)

    # (B,6,7,2) -> (B,42,2)
    x = layers.Reshape((6*7, 2))(inp)

    # project tokens
    x = layers.Dense(embed_dim)(x)

    # learned positional embedding
    pos = tf.range(start=0, limit=6*7, delta=1)
    pos_emb = layers.Embedding(input_dim=6*7, output_dim=embed_dim)(pos)
    x = x + pos_emb

    for _ in range(depth):
        x = TransformerBlock(embed_dim, num_heads, mlp_dim, dropout=0.1)(x)

    x = layers.LayerNormalization()(x)
    x = layers.GlobalAveragePooling1D()(x)

    x = layers.Dense(128, activation="relu")(x)
    x = layers.Dropout(0.35)(x)

    out = layers.Dense(7, activation="softmax", dtype="float32")(x)
    return keras.Model(inp, out, name="transformer_connect4")

trf = build_transformer()
trf.compile(
    optimizer=keras.optimizers.Adam(learning_rate=3e-4),
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)
trf.summary()

Model: "transformer_connect4"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer_6 (InputLayer)      │ (None, 6, 7, 2)        │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ reshape_1 (Reshape)             │ (None, 42, 2)          │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_13 (Dense)                │ (None, 42, 64)         │           192 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ add_1 (Add)                     │ (None, 42, 64)         │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ transformer_block_4             │ (None, 42, 64)         │        83,200 │
│ (TransformerBlock)              │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ transformer_block_5             │ (None, 42, 64)         │        83,200 │
│ (TransformerBlock)              │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ transformer_block_6             │ (None, 42, 64)         │        83,200 │
│ (TransformerBlock)              │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ transformer_block_7             │ (None, 42, 64)         │        83,200 │
│ (TransformerBlock)              │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ layer_normalization_17          │ (None, 42, 64)         │           128 │
│ (LayerNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling1d_1      │ (None, 64)             │             0 │
│ (GlobalAveragePooling1D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_22 (Dense)                │ (None, 128)            │         8,320 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_34 (Dropout)            │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_23 (Dense)                │ (None, 7)              │           903 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 342,343 (1.31 MB)

 Trainable params: 342,343 (1.31 MB)

 Non-trainable params: 0 (0.00 B)

## Train & Evaluate Transformer Model

In [14]:
callbacks_trf = [
    keras.callbacks.EarlyStopping(monitor="val_loss", patience=8, restore_best_weights=True),
    keras.callbacks.ReduceLROnPlateau(monitor="val_loss", factor=0.5, patience=3, min_lr=1e-6),
    keras.callbacks.ModelCheckpoint("best_transformer.keras", monitor="val_accuracy", save_best_only=True),
]

history_trf = trf.fit(
    train_ds,
    validation_data=val_ds,
    epochs=50,
    callbacks=callbacks_trf
)

trf_test = trf.evaluate(test_ds, verbose=0)
print("Transformer test loss/acc:", trf_test)

trf.save("transformer_final.keras")

Epoch 1/50
308/308 ━━━━━━━━━━━━━━━━━━━━ 82s 142ms/step - accuracy: 0.2780 - loss: 1.7704 - val_accuracy: 0.3547 - val_loss: 1.5624 - learning_rate: 3.0000e-04
Epoch 2/50
308/308 ━━━━━━━━━━━━━━━━━━━━ 3s 9ms/step - accuracy: 0.3487 - loss: 1.5705 - val_accuracy: 0.3828 - val_loss: 1.4828 - learning_rate: 3.0000e-04
Epoch 3/50
308/308 ━━━━━━━━━━━━━━━━━━━━ 3s 9ms/step - accuracy: 0.3773 - loss: 1.5043 - val_accuracy: 0.4073 - val_loss: 1.4529 - learning_rate: 3.0000e-04
Epoch 4/50
308/308 ━━━━━━━━━━━━━━━━━━━━ 3s 9ms/step - accuracy: 0.3927 - loss: 1.4785 - val_accuracy: 0.4149 - val_loss: 1.4416 - learning_rate: 3.0000e-04
Epoch 5/50
308/308 ━━━━━━━━━━━━━━━━━━━━ 3s 9ms/step - accuracy: 0.4011 - loss: 1.4646 - val_accuracy: 0.4206 - val_loss: 1.4216 - learning_rate: 3.0000e-04
Epoch 6/50
308/308 ━━━━━━━━━━━━━━━━━━━━ 3s 8ms/step - accuracy: 0.4050 - loss: 1.4521 - val_accuracy: 0.4187 - val_loss: 1.4216 - learning_rate: 3.0000e-04
Epoch 7/50
308/308 ━━━━━━━━━━━━━━━━━━━━ 3s 8ms/step - accurac

## Games against weak MCTS

In [16]:
import numpy as np
import random

ROWS, COLS = 6, 7

def canonicalize_xy2(board_xy2, player_to_move):
    if player_to_move == -1:
        return board_xy2[..., ::-1]
    return board_xy2

def legal_mask_from_board_xy2(board_6x7x2):
    top_filled = (board_6x7x2[0, :, 0] + board_6x7x2[0, :, 1]) > 0
    return np.where(top_filled, -1e9, 0.0)

def predict_move_legal(model, board_xy2, player_to_move=+1):
    b = canonicalize_xy2(board_xy2, player_to_move)
    probs = model.predict(b[None, ...], verbose=0)[0]
    logits = np.log(np.clip(probs, 1e-9, 1.0)) + legal_mask_from_board_xy2(b)
    return int(np.argmax(logits)), probs

def xy2_to_grid(board_xy2):
    grid = np.zeros((ROWS, COLS), dtype=np.int8)
    grid[board_xy2[...,0] == 1] = 1
    grid[board_xy2[...,1] == 1] = -1
    return grid

def grid_to_xy2(grid):
    b = np.zeros((ROWS, COLS, 2), dtype=np.int8)
    b[...,0] = (grid == 1).astype(np.int8)
    b[...,1] = (grid == -1).astype(np.int8)
    return b

def legal_moves(grid):
    return [c for c in range(COLS) if grid[0, c] == 0]

def apply_move(grid, col, player):
    if grid[0, col] != 0:
        return None
    g = grid.copy()
    for r in range(ROWS-1, -1, -1):
        if g[r, col] == 0:
            g[r, col] = player
            return g
    return None

def check_winner(grid):
    for r in range(ROWS):
        for c in range(COLS):
            p = grid[r, c]
            if p == 0:
                continue
            if c+3 < COLS and all(grid[r, c+i] == p for i in range(4)): return p
            if r+3 < ROWS and all(grid[r+i, c] == p for i in range(4)): return p
            if r+3 < ROWS and c+3 < COLS and all(grid[r+i, c+i] == p for i in range(4)): return p
            if r+3 < ROWS and c-3 >= 0 and all(grid[r+i, c-i] == p for i in range(4)): return p
    return 0

def is_draw(grid):
    return all(grid[0, c] != 0 for c in range(COLS))

def policy_random(grid, player):
    return random.choice(legal_moves(grid))

def rollout_random(grid, player):
    g = grid.copy()
    p = player
    while True:
        w = check_winner(g)
        if w != 0:
            return w
        if is_draw(g):
            return 0
        m = policy_random(g, p)
        g = apply_move(g, m, p)
        p = -p

def policy_weak_mcts(grid, player, sims=50):
    moves = legal_moves(grid)
    scores = {m: 0 for m in moves}
    for m in moves:
        g1 = apply_move(grid, m, player)
        if check_winner(g1) == player:
            return m
        for _ in range(sims):
            w = rollout_random(g1, -player)
            if w == player:
                scores[m] += 1
            elif w == -player:
                scores[m] -= 1
    return max(moves, key=lambda m: scores[m])

def play_game(model, opponent_fn, model_player=+1, opponent_sims=50):
    grid = np.zeros((ROWS, COLS), dtype=np.int8)
    player = +1
    while True:
        w = check_winner(grid)
        if w != 0:
            return w
        if is_draw(grid):
            return 0

        if player == model_player:
            b = grid_to_xy2(grid)
            move, _ = predict_move_legal(model, b, player)
        else:
            if opponent_fn == policy_weak_mcts:
                move = opponent_fn(grid, player, sims=opponent_sims)
            else:
                move = opponent_fn(grid, player)

        grid = apply_move(grid, move, player)
        player = -player

def eval_matchup(model, opponent_fn, games=50, model_player=+1, opponent_sims=50):
    results = {+1: 0, -1: 0, 0: 0}
    for _ in range(games):
        w = play_game(model, opponent_fn, model_player, opponent_sims)
        results[w] += 1
    return results

In [17]:
print("Vs random:", eval_matchup(cnn, policy_random, games=50))
print("Vs weak MCTS:", eval_matchup(cnn, policy_weak_mcts, games=30, opponent_sims=50))


Vs random: {1: 50, -1: 0, 0: 0}
Vs weak MCTS: {1: 26, -1: 2, 0: 2}


In [18]:
def summarize_results(results, model_player=+1, label="Matchup"):
    wins  = results[model_player]
    loss  = results[-model_player]
    draws = results[0]
    total = wins + loss + draws
    print(f"{label} (model plays as {model_player:+d})")
    print(f"  W-L-D: {wins}-{loss}-{draws}  (out of {total})")
    print(f"  Win%:  {wins/total:.1%} | Loss%: {loss/total:.1%} | Draw%: {draws/total:.1%}")

res1 = eval_matchup(cnn, policy_random, games=50, model_player=+1)
summarize_results(res1, model_player=+1, label="CNN vs Random")

res2 = eval_matchup(cnn, policy_weak_mcts, games=30, model_player=+1, opponent_sims=50)
summarize_results(res2, model_player=+1, label="CNN vs Weak MCTS (50 sims)")

CNN vs Random (model plays as +1)
  W-L-D: 50-0-0  (out of 50)
  Win%:  100.0% | Loss%: 0.0% | Draw%: 0.0%
CNN vs Weak MCTS (50 sims) (model plays as +1)
  W-L-D: 25-4-1  (out of 30)
  Win%:  83.3% | Loss%: 13.3% | Draw%: 3.3%


Illegal Move Masking, Move Prediction

In [ ]:
import numpy as np

def legal_mask_from_board_xy2(board_6x7x2):
    # top cell occupied if either channel has 1
    top_filled = (board_6x7x2[0, :, 0] + board_6x7x2[0, :, 1]) > 0
    return np.where(top_filled, -1e9, 0.0)

def predict_move(model, board_6x7x2):
    probs = model.predict(board_6x7x2[None, ...], verbose=0)[0]
    logits = np.log(np.clip(probs, 1e-9, 1.0)) + legal_mask_from_board_xy2(board_6x7x2)
    return int(np.argmax(logits)), probs

## Professor Mitchell's MCTS

In [19]:
import numpy as np

def update_board(board,color,col):
    # puts the checker on the board
    if color == 'plus':
        add = 1
    else:
        add = -1
    colsum = abs(board[0,col])+abs(board[1,col])+abs(board[2,col])+abs(board[3,col])+abs(board[4,col])+abs(board[5,col])
    row = int(5-colsum)
    board2 = board.copy()
    board2[row,col] = add
    return board2

def check_for_win(board,col):
    # faster win check (requires knowing last col played)
    nrow = 6
    ncol = 7
    colsum = abs(board[0,col])+abs(board[1,col])+abs(board[2,col])+abs(board[3,col])+abs(board[4,col])+abs(board[5,col])
    row = int(6-colsum)

    if row+3<6:
        vert = board[row,col] + board[row+1,col] + board[row+2,col] + board[row+3,col]
        if vert == 4:
            return 'v-plus'
        elif vert == -4:
            return 'v-minus'

    if col+3<7:
        hor = board[row,col] + board[row,col+1] + board[row,col+2] + board[row,col+3]
        if hor == 4:
            return 'h-plus'
        elif hor == -4:
            return 'h-minus'

    if col-1>=0 and col+2<7:
        hor = board[row,col-1] + board[row,col] + board[row,col+1] + board[row,col+2]
        if hor == 4:
            return 'h-plus'
        elif hor == -4:
            return 'h-minus'

    if col-2>=0 and col+1<7:
        hor = board[row,col-2] + board[row,col-1] + board[row,col] + board[row,col+1]
        if hor == 4:
            return 'h-plus'
        elif hor == -4:
            return 'h-minus'

    if col-3>=0:
        hor = board[row,col-3] + board[row,col-2] + board[row,col-1] + board[row,col]
        if hor == 4:
            return 'h-plus'
        elif hor == -4:
            return 'h-minus'

    # diag down-right
    if row+3<6 and col+3<7:
        diag = board[row,col] + board[row+1,col+1] + board[row+2,col+2] + board[row+3,col+3]
        if diag == 4:
            return 'd-plus'
        elif diag == -4:
            return 'd-minus'

    if row-1>=0 and row+2<6 and col-1>=0 and col+2<7:
        diag = board[row-1,col-1] + board[row,col] + board[row+1,col+1] + board[row+2,col+2]
        if diag == 4:
            return 'd-plus'
        elif diag == -4:
            return 'd-minus'

    if row-2>=0 and row+1<6 and col-2>=0 and col+1<7:
        diag = board[row-2,col-2] + board[row-1,col-1] + board[row,col] + board[row+1,col+1]
        if diag == 4:
            return 'd-plus'
        elif diag == -4:
            return 'd-minus'

    if row-3>=0 and col-3>=0:
        diag = board[row-3,col-3] + board[row-2,col-2] + board[row-1,col-1] + board[row,col]
        if diag == 4:
            return 'd-plus'
        elif diag == -4:
            return 'd-minus'

    # diag down-left
    if row+3<6 and col-3>=0:
        diag = board[row,col] + board[row+1,col-1] + board[row+2,col-2] + board[row+3,col-3]
        if diag == 4:
            return 'a-plus'
        elif diag == -4:
            return 'a-minus'

    if row-1>=0 and row+2<6 and col+1<7 and col-2>=0:
        diag = board[row-1,col+1] + board[row,col] + board[row+1,col-1] + board[row+2,col-2]
        if diag == 4:
            return 'a-plus'
        elif diag == -4:
            return 'a-minus'

    if row-2>=0 and row+1<6 and col+2<7 and col-1>=0:
        diag = board[row-2,col+2] + board[row-1,col+1] + board[row,col] + board[row+1,col-1]
        if diag == 4:
            return 'a-plus'
        elif diag == -4:
            return 'a-minus'

    if row-3>=0 and col+3<7:
        diag = board[row-3,col+3] + board[row-2,col+2] + board[row-1,col+1] + board[row,col]
        if diag == 4:
            return 'a-plus'
        elif diag == -4:
            return 'a-minus'

    return 'nobody'

def find_legal(board):
    legal = []
    for col in range(7):
        if board[0,col] == 0:
            legal.append(col)
    return legal

def look_for_win(board,color):
    legal = find_legal(board)
    for col in legal:
        b2 = update_board(board,color,col)
        winner = check_for_win(b2,col)
        if winner[2] == color[0]:
            return col
    return -1

def find_all_nonlosers(board,color):
    legal = find_legal(board)
    if color == 'plus':
        opp = 'minus'
    else:
        opp = 'plus'
    nonlosers = []
    for col in legal:
        b2 = update_board(board,color,col)
        # if opponent has an immediate win after this, it's a loser
        if look_for_win(b2,opp) < -0.5:
            nonlosers.append(col)
    return nonlosers

def back_prop(winner,path,color0,mcts_dict):
    # mcts_dict[state] = [visits, value_sum]
    for p in path:
        mcts_dict[p][0] += 1
        if winner == 'tie':
            mcts_dict[p][1] += 0
        elif winner[2] == color0[0]:
            mcts_dict[p][1] += 1
        else:
            mcts_dict[p][1] += -1

def rollout(board_temp,color0):
    board = board_temp.copy()
    color = color0
    winner = 'nobody'
    while winner == 'nobody':
        legal = find_legal(board)
        if len(legal) == 0:
            return 'tie'
        col = np.random.choice(legal)
        board = update_board(board,color,col)
        winner = check_for_win(board,col)
        if color == 'plus':
            color = 'minus'
        else:
            color = 'plus'
    return winner

def mcts(board_temp,color0,nsteps):
    board = board_temp.copy()

    # fast tactical pre-checks (prof code)
    winColumn = look_for_win(board,color0)
    if winColumn > -0.5:
        return winColumn
    legal0 = find_all_nonlosers(board,color0)
    if len(legal0) == 0:
        legal0 = find_legal(board)

    mcts_dict = {tuple(board.ravel()):[0,0]}
    for _ in range(nsteps):
        color = color0
        winner = 'nobody'
        board_mcts = board.copy()
        path = [tuple(board_mcts.ravel())]

        while winner == 'nobody':
            legal = find_legal(board_mcts)
            if len(legal) == 0:
                winner = 'tie'
                back_prop(winner,path,color0,mcts_dict)
                break

            board_list = []
            for col in legal:
                board_list.append(tuple(update_board(board_mcts,color,col).ravel()))
            for bl in board_list:
                if bl not in mcts_dict:
                    mcts_dict[bl] = [0,0]

            ucb1 = np.zeros(len(legal))
            for i in range(len(legal)):
                num_denom = mcts_dict[board_list[i]]
                if num_denom[0] == 0:
                    ucb1[i] = 10*nsteps
                else:
                    ucb1[i] = num_denom[1]/num_denom[0] + 2*np.sqrt(np.log(mcts_dict[path[-1]][0])/mcts_dict[board_list[i]][0])

            chosen = np.argmax(ucb1)
            board_mcts = update_board(board_mcts,color,legal[chosen])
            path.append(tuple(board_mcts.ravel()))
            winner = check_for_win(board_mcts,legal[chosen])

            if winner[2] == color[0]:
                back_prop(winner,path,color0,mcts_dict)
                break

            color = 'minus' if color == 'plus' else 'plus'

            if mcts_dict[tuple(board_mcts.ravel())][0] == 0:
                winner = rollout(board_mcts,color)
                back_prop(winner,path,color0,mcts_dict)
                break

    # choose best among "safe" moves
    maxval = -np.inf
    best_col = -1
    for col in legal0:
        board_temp2 = tuple(update_board(board,color0,col).ravel())
        num_denom = mcts_dict[board_temp2]
        compare = -np.inf if num_denom[0] == 0 else (num_denom[1] / num_denom[0])
        if compare > maxval:
            maxval = compare
            best_col = col
    return best_col

In [20]:
def policy_prof_mcts(grid, player, sims=2000):
    # player is +1 or -1
    color = 'plus' if player == 1 else 'minus'
    # professor MCTS expects a numpy array board with +1/-1/0
    return int(mcts(grid.astype(np.int8), color, int(sims)))

In [21]:
# moderate opponent
res = eval_matchup(cnn, policy_prof_mcts, games=20, model_player=+1, opponent_sims=500)
summarize_results(res, model_player=+1, label="CNN vs Prof MCTS (500 sims)")

# stronger opponent (slower)
res = eval_matchup(cnn, policy_prof_mcts, games=10, model_player=+1, opponent_sims=2000)
summarize_results(res, model_player=+1, label="CNN vs Prof MCTS (2000 sims)")

CNN vs Prof MCTS (500 sims) (model plays as +1)
  W-L-D: 12-3-5  (out of 20)
  Win%:  60.0% | Loss%: 15.0% | Draw%: 25.0%


KeyboardInterrupt: 